# V0 — End-to-End Retrieval Pipeline

Recipe embeddings via sentence-transformers, user embeddings as weighted means of liked recipes, cosine similarity retrieval, evaluated on a temporal holdout.

Dataset: Food.com (shuyangli94/food-com-recipes-and-user-interactions)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data")

## 1. Load and inspect data

In [2]:
recipes = pd.read_csv(DATA_DIR / "RAW_recipes.csv")
interactions = pd.read_csv(DATA_DIR / "RAW_interactions.csv")

print(f"Recipes: {len(recipes):,}")
print(f"Interactions: {len(interactions):,}")
print(f"Unique users: {interactions['user_id'].nunique():,}")
print(f"\nRecipe columns: {recipes.columns.tolist()}")
print(f"Interaction columns: {interactions.columns.tolist()}")
recipes.head(2)

Recipes: 231,637
Interactions: 1,132,367
Unique users: 226,570

Recipe columns: ['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags', 'nutrition', 'n_steps', 'steps', 'description', 'ingredients', 'n_ingredients']
Interaction columns: ['user_id', 'recipe_id', 'date', 'rating', 'review']


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6


In [3]:
# Check what tags look like — these are our concept vocabulary for V0
print(recipes['tags'].iloc[0])
print(type(recipes['tags'].iloc[0]))

['60-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'cuisine', 'preparation', 'occasion', 'north-american', 'side-dishes', 'vegetables', 'mexican', 'easy', 'fall', 'holiday-event', 'vegetarian', 'winter', 'dietary', 'christmas', 'seasonal', 'squash']
<class 'str'>


## 2. Prepare recipe text for embedding

Each recipe gets a single text string: name + ingredients + tags.
sentence-transformers will encode this into a dense vector.

In [4]:
import ast

def parse_list_str(s):
    """Parse a string representation of a list into an actual list."""
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return []

recipes['tags_list'] = recipes['tags'].apply(parse_list_str)
recipes['ingredients_list'] = recipes['ingredients'].apply(parse_list_str)

def build_recipe_text(row):
    """Combine name, ingredients, and tags into a single embedding input."""
    name = row['name']
    ingredients = ', '.join(row['ingredients_list'])
    tags = ', '.join(row['tags_list'])
    return f"{name}. Ingredients: {ingredients}. Tags: {tags}"

recipes['text'] = recipes.apply(build_recipe_text, axis=1)
print(recipes['text'].iloc[0])

arriba   baked winter squash mexican style. Ingredients: winter squash, mexican seasoning, mixed spice, honey, butter, olive oil, salt. Tags: 60-minutes-or-less, time-to-make, course, main-ingredient, cuisine, preparation, occasion, north-american, side-dishes, vegetables, mexican, easy, fall, holiday-event, vegetarian, winter, dietary, christmas, seasonal, squash


## 3. Embed recipes

Using `all-MiniLM-L6-v2` — small, fast, 384-dim, good enough for V0.
This takes a few minutes on ~230K recipes; we cache the result.

In [5]:
from sentence_transformers import SentenceTransformer

EMBEDDING_CACHE = DATA_DIR / "recipe_embeddings_v0.npy"
MODEL_NAME = "all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)

if EMBEDDING_CACHE.exists():
    print(f"Loading cached embeddings from {EMBEDDING_CACHE}")
    recipe_embeddings = np.load(EMBEDDING_CACHE)
else:
    print(f"Embedding {len(recipes):,} recipes...")
    recipe_embeddings = model.encode(
        recipes['text'].tolist(),
        show_progress_bar=True,
        batch_size=256,
        normalize_embeddings=True,
    )
    np.save(EMBEDDING_CACHE, recipe_embeddings)
    print(f"Saved to {EMBEDDING_CACHE}")

print(f"Shape: {recipe_embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading cached embeddings from ../data/recipe_embeddings_v0.npy
Shape: (231637, 384)


## 4. Temporal train/test split

Split interactions by date — train on earlier, evaluate on later.
This avoids the leakage you'd get from random splitting.

In [6]:
interactions['date'] = pd.to_datetime(interactions['date'])
interactions = interactions.sort_values('date')

# Use an 80/20 temporal split
split_idx = int(len(interactions) * 0.8)
train_interactions = interactions.iloc[:split_idx]
test_interactions = interactions.iloc[split_idx:]

print(f"Train: {len(train_interactions):,} interactions")
print(f"  Date range: {train_interactions['date'].min()} → {train_interactions['date'].max()}")
print(f"Test: {len(test_interactions):,} interactions")
print(f"  Date range: {test_interactions['date'].min()} → {test_interactions['date'].max()}")

Train: 905,893 interactions
  Date range: 2000-01-25 00:00:00 → 2011-12-27 00:00:00
Test: 226,474 interactions
  Date range: 2011-12-27 00:00:00 → 2018-12-20 00:00:00


In [7]:
# Filter to positive interactions (rating >= 4) — these are "likes"
# Rating of 0 means no rating given (just a review), treat as neutral
train_positive = train_interactions[train_interactions['rating'] >= 4].copy()
test_positive = test_interactions[test_interactions['rating'] >= 4].copy()

print(f"Positive train interactions: {len(train_positive):,}")
print(f"Positive test interactions: {len(test_positive):,}")

# Only evaluate users who appear in both train and test
common_users = set(train_positive['user_id']) & set(test_positive['user_id'])
print(f"Users in both train and test: {len(common_users):,}")

Positive train interactions: 822,501
Positive test interactions: 181,223
Users in both train and test: 10,000


## 5. Build user embeddings

Each user's embedding = normalized weighted mean of their positively-rated recipe embeddings.
Weights: rating 4 → weight 1, rating 5 → weight 2 (a 5-star pulls the embedding twice as hard).

In [8]:
recipe_id_to_idx = dict(zip(recipes['id'], range(len(recipes))))

RATING_WEIGHTS = {4: 1.0, 5: 2.0}

def build_user_embedding(recipe_ids, ratings):
    """Weighted mean of recipe embeddings, where 5-star recipes count double."""
    indices = []
    weights = []
    for rid, rating in zip(recipe_ids, ratings):
        if rid in recipe_id_to_idx:
            indices.append(recipe_id_to_idx[rid])
            weights.append(RATING_WEIGHTS.get(rating, 1.0))
    if not indices:
        return None
    weights = np.array(weights)
    emb = (recipe_embeddings[indices] * weights[:, None]).sum(axis=0) / weights.sum()
    emb = emb / np.linalg.norm(emb)
    return emb

In [9]:
# Build user embeddings from train set only
user_train_data = train_positive.groupby('user_id').agg(
    recipe_ids=('recipe_id', list),
    ratings=('rating', list),
)

user_embeddings = {}
user_seen_recipes = {}
for user_id in common_users:
    if user_id in user_train_data.index:
        row = user_train_data.loc[user_id]
        emb = build_user_embedding(row['recipe_ids'], row['ratings'])
        if emb is not None:
            user_embeddings[user_id] = emb
            user_seen_recipes[user_id] = set(row['recipe_ids'])

print(f"Built embeddings for {len(user_embeddings):,} users")

Built embeddings for 10,000 users


## 6. Retrieval via cosine similarity

For each user, find the top-k most similar recipes by dot product
(embeddings are already normalized, so dot product = cosine similarity).
231K vectors is small enough for numpy — no need for FAISS in V0.

In [10]:
K = 10

user_ids = list(user_embeddings.keys())
user_emb_matrix = np.array([user_embeddings[uid] for uid in user_ids], dtype=np.float32)
recipe_emb_matrix = recipe_embeddings.astype(np.float32)

BATCH = 500
recipe_ids_arr = recipes['id'].values
recommendations = {}

for start in range(0, len(user_ids), BATCH):
    batch_embs = user_emb_matrix[start:start + BATCH]
    sims = batch_embs @ recipe_emb_matrix.T

    for i in range(len(batch_embs)):
        uid = user_ids[start + i]
        seen = user_seen_recipes.get(uid, set())
        # Mask seen items before selecting top-K
        user_sims = sims[i].copy()
        for rid in seen:
            if rid in recipe_id_to_idx:
                user_sims[recipe_id_to_idx[rid]] = -np.inf
        top_k_idx = np.argpartition(-user_sims, K)[:K]
        sorted_idx = top_k_idx[np.argsort(-user_sims[top_k_idx])]
        recommendations[uid] = recipe_ids_arr[sorted_idx].tolist()

print(f"Generated top-{K} recommendations for {len(recommendations):,} users")

Generated top-10 recommendations for 10,000 users


## 7. Evaluation

Metrics on the temporal holdout:
- **Recall@K**: fraction of test-set liked recipes that appear in top-K
- **Hit Rate@K**: fraction of users who got at least one test-set hit in top-K
- **nDCG@K**: position-aware ranking quality

In [11]:
def recall_at_k(recommended, relevant):
    """What fraction of relevant items appear in the recommendation list?"""
    if not relevant:
        return 0.0
    return len(set(recommended) & set(relevant)) / len(relevant)

def hit_rate(recommended, relevant):
    """Did at least one relevant item appear?"""
    return 1.0 if set(recommended) & set(relevant) else 0.0

def ndcg_at_k(recommended, relevant):
    """Normalized discounted cumulative gain."""
    dcg = 0.0
    for i, item in enumerate(recommended):
        if item in relevant:
            dcg += 1.0 / np.log2(i + 2)  # i+2 because positions are 1-indexed
    # Ideal DCG: all relevant items at the top
    ideal_hits = min(len(relevant), len(recommended))
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0

In [12]:
# Build ground truth: for each user, what recipes did they like in the test set?
test_user_likes = test_positive.groupby('user_id')['recipe_id'].apply(set).to_dict()

recalls = []
hits = []
ndcgs = []

for uid in user_ids:
    relevant = test_user_likes.get(uid, set())
    if not relevant:
        continue
    recs = recommendations[uid]
    recalls.append(recall_at_k(recs, relevant))
    hits.append(hit_rate(recs, relevant))
    ndcgs.append(ndcg_at_k(recs, relevant))

print(f"Evaluated on {len(recalls):,} users")
print(f"")
print(f"Recall@{K}:   {np.mean(recalls):.4f}")
print(f"Hit Rate@{K}: {np.mean(hits):.4f}")
print(f"nDCG@{K}:     {np.mean(ndcgs):.4f}")

Evaluated on 10,000 users

Recall@10:   0.0009
Hit Rate@10: 0.0023
nDCG@10:     0.0007


## 8. Sanity checks

In [13]:
# Pick a random user and look at their recommendations vs history
sample_uid = user_ids[42]
sample_train_rids = list(user_seen_recipes[sample_uid])
print(f"User {sample_uid}")
print(f"\nLiked in train ({len(sample_train_rids)} recipes):")
for rid in sample_train_rids[:5]:
    if rid in recipe_id_to_idx:
        print(f"  - {recipes.iloc[recipe_id_to_idx[rid]]['name']}")

print(f"\nTop-{K} recommendations:")
for rid in recommendations[sample_uid]:
    idx = recipe_id_to_idx.get(rid)
    if idx is not None:
        print(f"  - {recipes.iloc[idx]['name']}")

test_likes = test_user_likes.get(sample_uid, set())
if test_likes:
    print(f"\nActually liked in test ({len(test_likes)} recipes):")
    for rid in list(test_likes)[:5]:
        if rid in recipe_id_to_idx:
            print(f"  - {recipes.iloc[recipe_id_to_idx[rid]]['name']}")

User 163986

Liked in train (64 recipes):
  - how to boil a lobster
  - easiest baked spam
  - tuna  red onion  and parsley salad
  - brownie fingers
  - perfect boiled eggs

Top-10 recommendations:
  - cheese and squeeze   cheddar and beef   biscuit balls
  - caramelized onion and white bean flatbread
  - sausage and roasted peppers pasta bake
  - onion bread pudding
  - cheddar and veggie bread pudding
  - spinach beef biscuit bake
  - wonderful beef and noodle casserole
  - firecracker casserole
  - sweety and sour meatballs
  - beefy biscuit casserole

Actually liked in test (2 recipes):
  - kittencal s taco seasoning mix
  - sweet cheese ball


In [14]:
# Verify seen-item filtering is working
leakage_counts = []
for uid in user_ids[:1000]:
    seen = user_seen_recipes.get(uid, set())
    recs = set(recommendations[uid])
    leakage_counts.append(len(recs & seen))

print(f"Mean train recipes in top-{K}: {np.mean(leakage_counts):.2f} (should be 0.00)")

Mean train recipes in top-10: 0.00 (should be 0.00)


## V0 results

See section 10 for the full comparison table with all baselines.

Protocol v1: global temporal 80/20 cut, 10K common users, 231,637-item catalog, K=10.
Seen-item filtering applied before candidate selection (bug fixed from earlier runs).

## 9. Experiment: max-sim retrieval (testing mean-collapse hypothesis)

Instead of scoring each candidate against the user's single mean embedding,
score it against *every* recipe the user liked and take the max.

If mean collapse is real, this should substantially outperform — it preserves
all of the user's distinct interests instead of averaging them into mush.

In [15]:
# Build per-user liked-recipe embedding matrices
user_liked_embs = {}
for uid in user_ids:
    seen = user_seen_recipes[uid]
    indices = [recipe_id_to_idx[rid] for rid in seen if rid in recipe_id_to_idx]
    if indices:
        user_liked_embs[uid] = recipe_embeddings[indices].astype(np.float32)

print(f"Built liked-recipe matrices for {len(user_liked_embs):,} users")
print(f"Example: user has {user_liked_embs[user_ids[0]].shape[0]} liked recipes, each {user_liked_embs[user_ids[0]].shape[1]}-dim")

Built liked-recipe matrices for 10,000 users
Example: user has 482 liked recipes, each 384-dim


In [16]:
K = 10
maxsim_recommendations = {}

for i, uid in enumerate(user_ids):
    if uid not in user_liked_embs:
        continue
    liked = user_liked_embs[uid]
    sims = liked @ recipe_emb_matrix.T
    max_sims = sims.max(axis=0)

    # Mask seen items before selecting top-K
    seen = user_seen_recipes.get(uid, set())
    for rid in seen:
        if rid in recipe_id_to_idx:
            max_sims[recipe_id_to_idx[rid]] = -np.inf

    top_k_idx = np.argpartition(-max_sims, K)[:K]
    sorted_idx = top_k_idx[np.argsort(-max_sims[top_k_idx])]
    maxsim_recommendations[uid] = recipe_ids_arr[sorted_idx].tolist()

    if (i + 1) % 2000 == 0:
        print(f"  {i + 1}/{len(user_ids)} users...")

print(f"Generated max-sim top-{K} for {len(maxsim_recommendations):,} users")

  2000/10000 users...
  4000/10000 users...
  6000/10000 users...
  8000/10000 users...
  10000/10000 users...
Generated max-sim top-10 for 10,000 users


In [17]:
maxsim_recalls = []
maxsim_hits = []
maxsim_ndcgs = []

for uid in user_ids:
    relevant = test_user_likes.get(uid, set())
    if not relevant or uid not in maxsim_recommendations:
        continue
    recs = maxsim_recommendations[uid]
    maxsim_recalls.append(recall_at_k(recs, relevant))
    maxsim_hits.append(hit_rate(recs, relevant))
    maxsim_ndcgs.append(ndcg_at_k(recs, relevant))

print(f"Evaluated on {len(maxsim_recalls):,} users")
print()
print(f"{'Method':<20} {'Recall@10':>10} {'HitRate@10':>12} {'nDCG@10':>10}")
print(f"{'-'*52}")
print(f"{'Mean embedding':<20} {np.mean(recalls):>10.4f} {np.mean(hits):>12.4f} {np.mean(ndcgs):>10.4f}")
print(f"{'Max-sim':<20} {np.mean(maxsim_recalls):>10.4f} {np.mean(maxsim_hits):>12.4f} {np.mean(maxsim_ndcgs):>10.4f}")

Evaluated on 10,000 users

Method                Recall@10   HitRate@10    nDCG@10
----------------------------------------------------
Mean embedding           0.0009       0.0023     0.0007
Max-sim                  0.0013       0.0053     0.0010


## 10. Reference frame — baselines

No method comparison is interpretable without these. Three baselines:
1. **Random** — expected recall from recommending 10 random unseen recipes
2. **Popularity** — recommend the 10 most-interacted recipes from train (that the user hasn't seen)
3. **Implicit ALS** — collaborative filtering on the interaction matrix (the behavioral signal our content model can't see)

In [18]:
# --- Random baseline ---
# Expected recall = K / catalog_size (minus seen items, but those are tiny relative to catalog)
n_catalog = len(recipes)
expected_random_recall = K / n_catalog
print(f"Random expected Recall@{K}: {expected_random_recall:.6f}")
print(f"  (= {K}/{n_catalog:,})")
print(f"  Our content model is ~{np.mean(recalls) / expected_random_recall:.0f}× random")

Random expected Recall@10: 0.000043
  (= 10/231,637)
  Our content model is ~20× random


In [19]:
# --- Popularity baseline ---
# Recommend the K most-interacted-with recipes from training (positive interactions only),
# filtering out recipes the user has already seen.

popularity = train_positive['recipe_id'].value_counts()
top_popular = popularity.index.tolist()  # sorted by count, descending

pop_recommendations = {}
for uid in user_ids:
    seen = user_seen_recipes.get(uid, set())
    recs = [rid for rid in top_popular if rid not in seen][:K]
    pop_recommendations[uid] = recs

# Evaluate
pop_recalls, pop_hits, pop_ndcgs = [], [], []
for uid in user_ids:
    relevant = test_user_likes.get(uid, set())
    if not relevant:
        continue
    recs = pop_recommendations[uid]
    pop_recalls.append(recall_at_k(recs, relevant))
    pop_hits.append(hit_rate(recs, relevant))
    pop_ndcgs.append(ndcg_at_k(recs, relevant))

print(f"Popularity baseline — evaluated on {len(pop_recalls):,} users")
print(f"  Recall@{K}:   {np.mean(pop_recalls):.4f}")
print(f"  HitRate@{K}:  {np.mean(pop_hits):.4f}")
print(f"  nDCG@{K}:     {np.mean(pop_ndcgs):.4f}")
print(f"\nTop-10 most popular recipes:")
for rid in top_popular[:10]:
    idx = recipe_id_to_idx.get(rid)
    if idx is not None:
        print(f"  {popularity[rid]:>5,} interactions  {recipes.iloc[idx]['name']}")

Popularity baseline — evaluated on 10,000 users
  Recall@10:   0.0204
  HitRate@10:  0.0608
  nDCG@10:     0.0139

Top-10 most popular recipes:
  1,125 interactions  to die for crock pot roast
  1,107 interactions  crock pot chicken with black beans   cream cheese
  1,024 interactions  creamy cajun chicken pasta
    952 interactions  whatever floats your boat  brownies
    832 interactions  jo mama s world famous spaghetti
    813 interactions  best ever banana cake with cream cheese frosting
    745 interactions  yes  virginia there is a great meatloaf
    745 interactions  kittencal s italian melt in your mouth meatballs
    711 interactions  oven fried chicken chimichangas
    689 interactions  japanese mum s chicken


In [20]:
# --- Implicit ALS (collaborative filtering) baseline ---
# This learns user and item latent factors from the interaction matrix alone —
# no content features, no text, no tags. Pure "users who liked X also liked Y."
#
# implicit uses the Alternating Least Squares algorithm for implicit feedback.
# We treat every positive interaction (rating >= 4) as a "1" in the interaction matrix.

from scipy.sparse import csr_matrix
import implicit

# Build user/item ID → contiguous index mappings for the interaction matrix
train_user_ids = train_positive['user_id'].unique()
train_recipe_ids = train_positive['recipe_id'].unique()

user_id_map = {uid: i for i, uid in enumerate(train_user_ids)}
item_id_map = {rid: i for i, rid in enumerate(train_recipe_ids)}
item_id_reverse = {i: rid for rid, i in item_id_map.items()}

# Build sparse interaction matrix (users × items)
rows = train_positive['user_id'].map(user_id_map).values
cols = train_positive['recipe_id'].map(item_id_map).values
data = np.ones(len(train_positive), dtype=np.float32)

user_item_matrix = csr_matrix(
    (data, (rows, cols)),
    shape=(len(train_user_ids), len(train_recipe_ids)),
)

print(f"Interaction matrix: {user_item_matrix.shape}")
print(f"  Non-zeros: {user_item_matrix.nnz:,}")
print(f"  Density: {user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1]):.6f}")

Interaction matrix: (109103, 194151)
  Non-zeros: 822,501
  Density: 0.000039


In [21]:
# Train ALS model
# factors=64 is a reasonable starting point; iterations=15 is standard
als_model = implicit.als.AlternatingLeastSquares(
    factors=64,
    iterations=15,
    regularization=0.01,
    random_state=42,
)
als_model.fit(user_item_matrix)
print("ALS model trained")

  0%|          | 0/15 [00:00<?, ?it/s]

ALS model trained


In [22]:
# Generate ALS recommendations for eval users
# implicit's recommend() already filters seen items
als_recommendations = {}

for uid in user_ids:
    if uid not in user_id_map:
        continue
    uidx = user_id_map[uid]
    # recommend returns (item_indices, scores)
    item_indices, scores = als_model.recommend(
        uidx,
        user_item_matrix[uidx],
        N=K,
        filter_already_liked_items=True,
    )
    als_recommendations[uid] = [item_id_reverse[i] for i in item_indices]

print(f"Generated ALS top-{K} for {len(als_recommendations):,} users")

# Evaluate
als_recalls, als_hits, als_ndcgs = [], [], []
for uid in user_ids:
    relevant = test_user_likes.get(uid, set())
    if not relevant or uid not in als_recommendations:
        continue
    recs = als_recommendations[uid]
    als_recalls.append(recall_at_k(recs, relevant))
    als_hits.append(hit_rate(recs, relevant))
    als_ndcgs.append(ndcg_at_k(recs, relevant))

print(f"ALS baseline — evaluated on {len(als_recalls):,} users")
print(f"  Recall@{K}:   {np.mean(als_recalls):.4f}")
print(f"  HitRate@{K}:  {np.mean(als_hits):.4f}")
print(f"  nDCG@{K}:     {np.mean(als_ndcgs):.4f}")

Generated ALS top-10 for 10,000 users
ALS baseline — evaluated on 10,000 users
  Recall@10:   0.0140
  HitRate@10:  0.0499
  nDCG@10:     0.0108


In [23]:
# --- Full comparison table ---
print(f"Protocol v1 — global temporal cut, 10K common users, 231,637-item catalog, K={K}")
print(f"Binomial SE at p=0.003, n=10000: ±{np.sqrt(0.003 * 0.997 / 10000):.4f}")
print()
print(f"{'Method':<25} {'Recall@10':>10} {'HitRate@10':>12} {'nDCG@10':>10}  {'n_hits':>7}")
print(f"{'-'*68}")
print(f"{'Random (expected)':<25} {expected_random_recall:>10.6f} {'—':>12} {'—':>10}  {'—':>7}")
print(f"{'Popularity':<25} {np.mean(pop_recalls):>10.4f} {np.mean(pop_hits):>12.4f} {np.mean(pop_ndcgs):>10.4f}  {sum(1 for h in pop_hits if h > 0):>7,}")
print(f"{'Content mean':<25} {np.mean(recalls):>10.4f} {np.mean(hits):>12.4f} {np.mean(ndcgs):>10.4f}  {sum(1 for h in hits if h > 0):>7,}")
print(f"{'Content max-sim':<25} {np.mean(maxsim_recalls):>10.4f} {np.mean(maxsim_hits):>12.4f} {np.mean(maxsim_ndcgs):>10.4f}  {sum(1 for h in maxsim_hits if h > 0):>7,}")
print(f"{'Implicit ALS (CF)':<25} {np.mean(als_recalls):>10.4f} {np.mean(als_hits):>12.4f} {np.mean(als_ndcgs):>10.4f}  {sum(1 for h in als_hits if h > 0):>7,}")

Protocol v1 — global temporal cut, 10K common users, 231,637-item catalog, K=10
Binomial SE at p=0.003, n=10000: ±0.0005

Method                     Recall@10   HitRate@10    nDCG@10   n_hits
--------------------------------------------------------------------
Random (expected)           0.000043            —          —        —
Popularity                    0.0204       0.0608     0.0139      608
Content mean                  0.0009       0.0023     0.0007       23
Content max-sim               0.0013       0.0053     0.0010       53
Implicit ALS (CF)             0.0140       0.0499     0.0108      499


## 11. Sampled-negative ranking eval

Full-catalog retrieval asks "can you find the needle in a 231K haystack?" —
dominated by popularity and exposure. That's not the right question for a taste model.

Sampled-negative ranking asks: **given a recipe this user actually liked, can the model
rank it above 100 random recipes the user didn't interact with?**

This directly measures whether the representation encodes preference order.
A content model that's "semantically coherent but doesn't predict clicks" might
still rank liked items above random ones — which is exactly what matters for
cold start, onboarding, and user-user matching.

In [24]:
N_NEGATIVES = 100
rng = np.random.RandomState(42)

# For each user, for each test-positive recipe, sample 100 random unseen negatives
all_recipe_ids = set(recipes['id'].values)

eval_triples = []  # (user_id, positive_recipe_id, [negative_recipe_ids])
for uid in user_ids:
    relevant = test_user_likes.get(uid, set())
    if not relevant:
        continue
    seen = user_seen_recipes.get(uid, set())
    # Pool of candidates for negatives: everything the user hasn't interacted with
    # (in train OR test positive)
    exclude = seen | relevant
    neg_pool = list(all_recipe_ids - exclude)
    
    for pos_rid in relevant:
        if pos_rid not in recipe_id_to_idx:
            continue
        negs = rng.choice(neg_pool, size=N_NEGATIVES, replace=False).tolist()
        eval_triples.append((uid, pos_rid, negs))

print(f"Eval triples: {len(eval_triples):,} (user, positive, 100 negatives)")

Eval triples: 90,075 (user, positive, 100 negatives)


In [28]:
def sampled_neg_eval(score_fn, label):
    """Run sampled-negative ranking eval.
    
    score_fn(user_id, recipe_ids) → array of scores, one per recipe_id.
    For each triple, rank the positive among 101 items. Report AUC and HR@10.
    """
    aucs = []
    hr10s = []
    
    for uid, pos_rid, neg_rids in eval_triples:
        all_rids = [pos_rid] + neg_rids
        scores = score_fn(uid, all_rids)
        if scores is None:
            continue
        
        pos_score = scores[0]
        neg_scores = scores[1:]
        
        # AUC = fraction of negatives the positive beats
        auc = np.mean(pos_score > neg_scores) + 0.5 * np.mean(pos_score == neg_scores)
        aucs.append(auc)
        
        # HR@10 = is the positive in the top 10?
        rank = 1 + np.sum(neg_scores > pos_score)  # 1-indexed rank
        hr10s.append(1.0 if rank <= 10 else 0.0)
    
    print(f"{label:<25} AUC={np.mean(aucs):.4f}  HR@10={np.mean(hr10s):.4f}  (n={len(aucs):,})")
    return {'method': label, 'auc': np.mean(aucs), 'hr10': np.mean(hr10s), 'n': len(aucs)}

In [29]:
# --- Scoring functions for each method ---

def content_mean_score(uid, recipe_ids):
    """Cosine similarity between user mean embedding and each recipe."""
    if uid not in user_embeddings:
        return None
    user_emb = user_embeddings[uid].astype(np.float32)
    indices = [recipe_id_to_idx[rid] for rid in recipe_ids if rid in recipe_id_to_idx]
    if len(indices) != len(recipe_ids):
        return None
    embs = recipe_emb_matrix[indices]
    return embs @ user_emb

def content_maxsim_score(uid, recipe_ids):
    """Max cosine similarity to any liked recipe."""
    if uid not in user_liked_embs:
        return None
    liked = user_liked_embs[uid]
    indices = [recipe_id_to_idx[rid] for rid in recipe_ids if rid in recipe_id_to_idx]
    if len(indices) != len(recipe_ids):
        return None
    embs = recipe_emb_matrix[indices]  # (101 x 384)
    sims = embs @ liked.T              # (101 x n_liked)
    return sims.max(axis=1)            # (101,)

def popularity_score(uid, recipe_ids):
    """Score = number of positive interactions in train."""
    return np.array([popularity.get(rid, 0) for rid in recipe_ids], dtype=np.float32)

def als_score(uid, recipe_ids):
    """Score = dot product of ALS user/item factors."""
    if uid not in user_id_map:
        return None
    uidx = user_id_map[uid]
    user_factors = als_model.user_factors[uidx]
    scores = []
    for rid in recipe_ids:
        if rid in item_id_map:
            scores.append(np.dot(user_factors, als_model.item_factors[item_id_map[rid]]))
        else:
            scores.append(0.0)  # cold item — ALS has no factor for it
    return np.array(scores, dtype=np.float32)

def random_score(uid, recipe_ids):
    """Random scores — the AUC should be ~0.50."""
    return rng.rand(len(recipe_ids)).astype(np.float32)

In [30]:
# --- Run all methods ---
print(f"Sampled-negative ranking: each test positive vs {N_NEGATIVES} random negatives")
print(f"{'Method':<25} {'AUC':>8}  {'HR@10':>8}  {'n':>8}")
print(f"{'-'*55}")

results_sn = []
results_sn.append(sampled_neg_eval(random_score, 'Random'))
results_sn.append(sampled_neg_eval(popularity_score, 'Popularity'))
results_sn.append(sampled_neg_eval(content_mean_score, 'Content mean'))
results_sn.append(sampled_neg_eval(content_maxsim_score, 'Content max-sim'))
results_sn.append(sampled_neg_eval(als_score, 'Implicit ALS (CF)'))

Sampled-negative ranking: each test positive vs 100 random negatives
Method                         AUC     HR@10         n
-------------------------------------------------------
Random                    AUC=0.4998  HR@10=0.0980  (n=90,075)
Popularity                AUC=0.6156  HR@10=0.4091  (n=90,075)
Content mean              AUC=0.5711  HR@10=0.1462  (n=90,075)
Content max-sim           AUC=0.6193  HR@10=0.2081  (n=90,075)
Implicit ALS (CF)         AUC=0.6172  HR@10=0.3144  (n=90,075)
